# 01 · CAMELYON17: eksploracja danych i przygotowanie masek

Co robi ten notebook:
1. opisuje zbiór (liczba slajdów, pacjentów, ośrodków, rozmiary i piramidy WSI, etykiety stage),
2. pokazuje krok po kroku, jak jeden plik XML z poligonami zmian przerzutowych staje się maską guza,
3. robi to samo dla wszystkich zaadnotowanych slajdów (`preprocess()`) i sprawdza błędy w adnotacjach,
4. opisuje rozkłady (obszary guza, liczba patchy na klasę, różnice barwienia między ośrodkami) oraz zamrożony podział train/val **po pacjencie**.

Założenia: oryginalny **CAMELYON17** (całe skany WSI `.tif` + adnotacje ASAP `.xml` + `stage_labels.csv`),
a nie wersja patchowa z WILDS. Ścieżki ustaw w komórce konfiguracji.

In [ ]:
# --- działa bez instalacji pakietu: dodaje src/ repo do ścieżki importu ---
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "histopathology_pnw").exists())
sys.path.insert(0, str(_root / "src"))

In [ ]:
import json
import math
import os
import re
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import histopathology_pnw

try:  # biblioteka libopenslide z pakietu openslide-bin (gdy nie ma jej w systemie)
    import openslide_bin  # noqa: F401
except ImportError:
    pass
import openslide
import pandas as pd
from skimage.color import label2rgb, rgb2hsv
from skimage.draw import polygon as draw_polygon
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_holes, remove_small_objects
from skimage.segmentation import find_boundaries

pd.set_option("display.width", 160)

# --- konfiguracja: dostosuj do swojego układu katalogów ---
ROOT = Path(histopathology_pnw.__file__).resolve().parents[2]  # korzeń repo, niezależnie od katalogu notebooka
DATA_ROOT = Path(os.environ.get("C17_ROOT", ROOT / "data" / "raw" / "camelyon17"))  # na WCSS: $PD/.../camelyon17/raw
IMAGES = DATA_ROOT / "images"                 # rekurencyjnie *.tif (np. training/center_0/...)
ANNOT = DATA_ROOT / "lesion_annotations"      # *.xml w formacie ASAP
STAGES = DATA_ROOT / "stage_labels.csv"
OUT = ROOT / "data" / "processed" / "camelyon17"
SPLIT_PATH = ROOT / "configs" / "camelyon17_split.json"
MASK_LEVEL = 5      # poziom piramidy, na którym budujemy maski (downsample ~32x)
PATCH = 256         # rozmiar patcha na poziomie 0 [px]
OUT.mkdir(parents=True, exist_ok=True)

SLIDE_RE = re.compile(r"patient_(\d+)_node_(\d+)")


@dataclass
class Slide:
    slide_id: str
    patient: int
    node: int
    center: int          # 20 pacjentów na ośrodek: pacjenci 0-19 -> ośrodek 0, itd.
    image_path: Path
    xml_path: Path | None


def list_slides():
    xmls = {p.stem: p for p in ANNOT.glob("*.xml")}
    slides = []
    for p in sorted(IMAGES.rglob("*.tif")):
        m = SLIDE_RE.search(p.stem)
        if not m:
            continue
        patient, node = int(m.group(1)), int(m.group(2))
        slides.append(Slide(p.stem, patient, node, patient // 20, p, xmls.get(p.stem)))
    return slides


# --- pobieranie danych (dociąga brakujące; już pobrane pliki są pomijane) ---
DOWNLOAD = os.environ.get("C17_DOWNLOAD", "1") == "1"  # na węźle obliczeniowym ustaw C17_DOWNLOAD=0 (brak internetu)
MAX_SLIDES = 10  # ile zaadnotowanych slajdów ma być na dysku; zwiększ, żeby dorzucić kolejne. None = wszystkie 50 (ok. 140 GB)
CENTERS = None  # None = wszystkie ośrodki; np. [0, 1] = tylko ośrodki 0 i 1 (ośrodek = pacjent // 20)
ONLY_ANNOTATED = True  # True: tylko slajdy z XML (potrzebne do notebooków 01-03); False: wszystkie (setki GB)
if DOWNLOAD:
    try:
        from histopathology_pnw.camelyon17_download import download_camelyon17
    except ImportError:  # plik mógł trafić do datasets/
        from histopathology_pnw.datasets.camelyon17_download import download_camelyon17

    have = sum(1 for s in list_slides() if s.xml_path is not None)
    if have < (MAX_SLIDES or 50):
        print(f"na dysku: {have} slajdów z XML, docelowo: {MAX_SLIDES or 'wszystkie'}")
        download_camelyon17(DATA_ROOT, only_annotated=ONLY_ANNOTATED, max_slides=MAX_SLIDES, centers=CENTERS)

slides = list_slides()
annotated = [s for s in slides if s.xml_path is not None]
orphan_xml = sorted(set(p.stem for p in ANNOT.glob("*.xml")) - {s.slide_id for s in slides})
print(f"slajdy: {len(slides)}, z adnotacją XML: {len(annotated)}, XML bez slajdu: {len(orphan_xml)}")

## 1. Zbiór w liczbach

CAMELYON17 (Bándi et al. 2019): skany WSI węzłów chłonnych z piersi (barwienie H&E) z **5 ośrodków**
(po 20 pacjentów na ośrodek w zbiorze treningowym, po 5 węzłów na pacjenta, czyli 500 slajdów).
Tylko część slajdów treningowych ma ręcznie obrysowane zmiany przerzutowe. Pozostałe mają wyłącznie etykietę
slajdu (`negative` / `itc` / `micro` / `macro`) i etykietę pacjenta (pN-stage).

Każdy slajd to piramida obrazów (poziom 0 = pełna rozdzielczość, kolejne poziomy pomniejszone),
dlatego nie wczytujemy go w całości, tylko przez `openslide`.

In [ ]:
rows = []
for s in slides:
    row = dict(slide_id=s.slide_id, patient=s.patient, node=s.node, center=s.center,
               annotated=s.xml_path is not None)
    try:
        with openslide.OpenSlide(str(s.image_path)) as sl:
            row.update(
                width=sl.dimensions[0], height=sl.dimensions[1], levels=sl.level_count,
                mpp=float(sl.properties.get(openslide.PROPERTY_NAME_MPP_X, "nan")),
                vendor=sl.properties.get(openslide.PROPERTY_NAME_VENDOR),
            )
    except Exception as e:  # uszkodzony plik, brak wsparcia formatu itp.
        row["error"] = repr(e)
    rows.append(row)
info = pd.DataFrame(rows)
if "error" in info:
    print("slajdy, których nie udało się otworzyć:", info["error"].notna().sum())

info.groupby("center").agg(
    slajdy=("slide_id", "count"),
    pacjenci=("patient", "nunique"),
    z_adnotacja=("annotated", "sum"),
    mpp_mediana=("mpp", "median"),
    szer_mediana=("width", "median"),
    wys_mediana=("height", "median"),
)

In [ ]:
print("liczba poziomów piramidy:", info["levels"].value_counts().to_dict())
print("producenci skanerów (vendor):", info["vendor"].value_counts(dropna=False).to_dict())
info[["width", "height"]].describe().T

Etykiety slajdów i pacjentów z `stage_labels.csv`. Plik zawiera wiersze dwóch typów:
`patient_XXX_node_Y.tif` (etykieta slajdu) i `patient_XXX.zip` (stage pN pacjenta).

In [ ]:
if STAGES.exists():
    st = pd.read_csv(STAGES)
    st.columns = ["name", "label"] + list(st.columns[2:])
    slide_labels = (st[st["name"].str.endswith(".tif")]
                    .assign(slide_id=lambda d: d["name"].str.replace(".tif", "", regex=False))
                    [["slide_id", "label"]])
    patient_labels = st[~st["name"].str.endswith(".tif")]
    print("etykiety slajdów:", slide_labels["label"].value_counts().to_dict())
    print("etykiety pacjentów (pN):", patient_labels["label"].value_counts().to_dict())
else:
    # plik jest opcjonalny: bez niego pomijamy tylko etykiety slajdów (negative/itc/micro/macro)
    print(f"brak {STAGES}: pomijam etykiety slajdów (reszta notebooka działa bez nich)")
    slide_labels = pd.DataFrame({"slide_id": pd.Series(dtype=str), "label": pd.Series(dtype=str)})
    patient_labels = slide_labels
info = info.merge(slide_labels, on="slide_id", how="left")
pd.crosstab(info["center"], info["label"].fillna("brak"), margins=True)

## 2. Format adnotacji: XML (ASAP) z poligonami

Każda zmiana to `Annotation` z atrybutami `Name`, `Type` (zwykle `Polygon`) i `PartOfGroup`, a w środku
lista `Coordinate Order=… X=… Y=…`. Współrzędne są w pikselach **poziomu 0** (zmiennoprzecinkowe).
Grupa `metastases` oznacza guz, a `normal` to obszary zdrowe wewnątrz obrysu guza (dziury). Hierarchia:
`ASAP_Annotations/Annotations/Annotation/Coordinates/Coordinate`.

In [ ]:
sample = annotated[0]
root = ET.parse(sample.xml_path).getroot()
ann = next(root.iter("Annotation"))
coords = list(ann.iter("Coordinate"))
print("plik:", sample.xml_path.name)
print("element główny:", root.tag, "| pod nim:", [c.tag for c in root])
print("liczba adnotacji:", sum(1 for _ in root.iter("Annotation")))
print("atrybuty adnotacji:", {k: ann.attrib.get(k) for k in ("Name", "Type", "PartOfGroup")})
print("liczba wierzchołków pierwszej:", len(coords))
print("pierwsze wierzchołki:", [c.attrib for c in coords[:3]])

# rozkład typów i grup w całym zbiorze adnotacji
meta = pd.DataFrame([
    dict(slide_id=s.slide_id, type=a.attrib.get("Type"), group=a.attrib.get("PartOfGroup"))
    for s in annotated for a in ET.parse(s.xml_path).getroot().iter("Annotation")
])
meta.groupby(["type", "group"]).size()

## 3. Jeden slajd krok po kroku: XML → maska guza

1. `read_polygons` czyta poligony jako tablice `(N, 2)` współrzędnych `(x, y)` na poziomie 0 (wierzchołki sortowane po `Order`).
2. `polygons_to_masks` skaluje współrzędne przez `downsample` wybranego poziomu i rasteryzuje je
   `skimage.draw.polygon` (wiersz = y, kolumna = x). Najpierw rysowane są poligony guza (`metastases`),
   potem grupa `normal` wycina z nich dziury. Poligony z < 3 wierzchołkami są odrzucane,
   a wierzchołki poza obrazem przycinane.
3. `tissue_mask` wyznacza tkankę progiem Otsu na kanale saturacji (HSV), żeby odróżnić tkankę od białego tła szkiełka.
4. Wynik: dwie maski `bool` na poziomie `MASK_LEVEL`: guz i tkanka. Pełnej rozdzielczości nie trzymamy,
   a patche na poziomie 0 wycina się później po współrzędnych.

In [ ]:
def read_polygons(xml_path):
    # zwraca listę (group, type, array (N, 2) w pikselach poziomu 0)
    out = []
    for a in ET.parse(xml_path).getroot().iter("Annotation"):
        cs = sorted(a.iter("Coordinate"), key=lambda c: float(c.attrib.get("Order", 0)))
        xy = np.array([[float(c.attrib["X"]), float(c.attrib["Y"])] for c in cs]).reshape(-1, 2)
        out.append((a.attrib.get("PartOfGroup", "None").lower(), a.attrib.get("Type"), xy))
    return out


def polygons_to_masks(polys, shape, downsample, level0_size):
    # shape = (h, w) maski na wybranym poziomie, level0_size = (W, H) poziomu 0
    tumor = np.zeros(shape, bool)
    normal = np.zeros(shape, bool)
    stats = dict(n_polygons=len(polys), n_too_few_vertices=0, n_metastases=0, n_normal=0,
                 n_out_of_bounds=0, n_empty=0)
    W, H = level0_size
    for group, _typ, xy in polys:
        if len(xy) < 3:
            stats["n_too_few_vertices"] += 1
            continue
        if (xy[:, 0] < 0).any() or (xy[:, 1] < 0).any() or (xy[:, 0] > W).any() or (xy[:, 1] > H).any():
            stats["n_out_of_bounds"] += 1
        rr, cc = draw_polygon(xy[:, 1] / downsample, xy[:, 0] / downsample, shape)
        if len(rr) == 0:
            stats["n_empty"] += 1
            continue
        if group == "normal":
            normal[rr, cc] = True
            stats["n_normal"] += 1
        else:
            tumor[rr, cc] = True
            stats["n_metastases"] += 1
    return tumor & ~normal, stats


def tissue_mask(rgb):
    sat = rgb2hsv(rgb)[..., 1]
    m = sat > threshold_otsu(sat)
    m = remove_small_objects(m, 64)
    return remove_small_holes(m, 64)


def read_level(slide, level):
    w, h = slide.level_dimensions[level]
    return np.asarray(slide.read_region((0, 0), level, (w, h)).convert("RGB"))

In [ ]:
with openslide.OpenSlide(str(sample.image_path)) as sl:
    level = min(MASK_LEVEL, sl.level_count - 1)
    ds = sl.level_downsamples[level]
    thumb = read_level(sl, level)
    polys = read_polygons(sample.xml_path)
    tumor, stats = polygons_to_masks(polys, thumb.shape[:2], ds, sl.dimensions)
    tissue = tissue_mask(thumb)
    print(f"{sample.slide_id}: poziom {level}, downsample {ds:.1f}, rozmiar {thumb.shape[:2]}")
    print(stats)

    # wycinek w wyższej rozdzielczości wokół pierwszego guza (poziom 2, 1024x1024 px)
    zl = min(2, sl.level_count - 1)
    zds = sl.level_downsamples[zl]
    big = max((p for g, _, p in polys if g != "normal"), key=len)
    cx, cy = big.mean(axis=0)
    x0 = int(max(cx - 512 * zds, 0)); y0 = int(max(cy - 512 * zds, 0))
    zoom = np.asarray(sl.read_region((x0, y0), zl, (1024, 1024)).convert("RGB"))

fig, ax = plt.subplots(1, 4, figsize=(22, 5.5))
ax[0].imshow(thumb); ax[0].set_title(f"slajd {sample.slide_id} (poziom {level})")
ax[1].imshow(thumb)
for g, _, p in polys:
    ax[1].plot(p[:, 0] / ds, p[:, 1] / ds, lw=1, color="red" if g != "normal" else "cyan")
ax[1].set_title("poligony z XML (czerwone = guz, cyjan = normal)")
ax[2].imshow(label2rgb(tumor.astype(int) + 2 * (tissue & ~tumor), image=thumb, bg_label=0, alpha=0.5,
                       colors=["red", "limegreen"]))
ax[2].set_title("maska: guz (czerwony), reszta tkanki (zielony)")
ov = zoom.copy()
for g, _, p in polys:
    q = (p - [x0, y0]) / zds
    ax[3].plot(q[:, 0], q[:, 1], lw=1.2, color="lime" if g != "normal" else "cyan")
ax[3].imshow(ov); ax[3].set_xlim(0, 1024); ax[3].set_ylim(1024, 0)
ax[3].set_title(f"wycinek 1024 px, poziom {zl}, z konturami")
for a in ax:
    a.axis("off")
plt.tight_layout()

## 4. Wszystkie zaadnotowane slajdy

`preprocess()` robi to samo dla każdego slajdu z XML, zapisuje `data/processed/camelyon17/<id>.npz`
(maski `tumor` i `tissue`, `downsample`) oraz `manifest.csv` ze statystykami. Dodatkowo liczy średni kolor tkanki
(potrzebny do porównania barwienia między ośrodkami).

In [ ]:
def preprocess(slides_with_xml, force=False):
    rows = []
    for s in slides_with_xml:
        out_path = OUT / f"{s.slide_id}.npz"
        with openslide.OpenSlide(str(s.image_path)) as sl:
            level = min(MASK_LEVEL, sl.level_count - 1)
            ds = float(sl.level_downsamples[level])
            thumb = read_level(sl, level)
            polys = read_polygons(s.xml_path)
            tumor, stats = polygons_to_masks(polys, thumb.shape[:2], ds, sl.dimensions)
            tissue = tissue_mask(thumb)
            np.savez_compressed(out_path, tumor=tumor, tissue=tissue, downsample=ds, level=level)
            mean_rgb = thumb[tissue].mean(axis=0) if tissue.any() else np.full(3, np.nan)
            rows.append(dict(
                slide_id=s.slide_id, patient=s.patient, node=s.node, center=s.center,
                mask_level=level, downsample=ds, **stats,
                tissue_px=int(tissue.sum()), tumor_px=int(tumor.sum()),
                tumor_outside_tissue_px=int((tumor & ~tissue).sum()),
                tumor_frac_of_tissue=float(tumor.sum() / max(tissue.sum(), 1)),
                mean_r=mean_rgb[0], mean_g=mean_rgb[1], mean_b=mean_rgb[2],
            ))
    manifest = pd.DataFrame(rows)
    manifest.to_csv(OUT / "manifest.csv", index=False)
    return manifest


manifest = preprocess(annotated)
manifest[["n_polygons", "n_metastases", "n_normal", "n_too_few_vertices", "n_out_of_bounds", "n_empty"]].sum()

### Błędy i niespójności w adnotacjach

Slajdy, w których coś zostało odrzucone, wyszło poza obraz lub guz leży poza wykrytą tkanką
(co zwykle znaczy, że maska tkanki jest zbyt zachowawcza albo poligon jest przesunięty). Sprawdzamy też
zgodność z `stage_labels.csv`: slajd `negative` nie powinien mieć żadnego poligonu guza,
a `micro`/`macro` powinny.

In [ ]:
issues = manifest[
    (manifest[["n_too_few_vertices", "n_empty", "n_out_of_bounds"]].sum(axis=1) > 0)
    | (manifest["tumor_outside_tissue_px"] > 0.05 * manifest["tumor_px"].clip(lower=1))
]
print(f"slajdy z jakimkolwiek problemem: {len(issues)} / {len(manifest)}")
display(issues.sort_values("tumor_outside_tissue_px", ascending=False)
        [["slide_id", "center", "n_too_few_vertices", "n_out_of_bounds", "n_empty",
          "tumor_px", "tumor_outside_tissue_px"]])

chk = manifest.merge(slide_labels, on="slide_id", how="left")
chk["ma_guz_w_xml"] = chk["n_metastases"] > 0
pd.crosstab(chk["label"].fillna("brak"), chk["ma_guz_w_xml"])

## 5. Rozmiary zmian

Liczba poligonów guza na slajd i powierzchnia guza (w mm², z `mpp` slajdu i `downsample` maski).
Zmiany `micro` i `macro` różnią się o rzędy wielkości, co ma znaczenie dla balansu klas.

In [ ]:
m = manifest.merge(info[["slide_id", "mpp"]], on="slide_id")
m["tumor_mm2"] = m["tumor_px"] * (m["downsample"] * m["mpp"] / 1000) ** 2
m["tissue_mm2"] = m["tissue_px"] * (m["downsample"] * m["mpp"] / 1000) ** 2

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].hist(m["n_metastases"], bins=20); ax[0].set_xlabel("poligonów guza na slajd")
ax[1].hist(np.log10(m["tumor_mm2"].clip(lower=1e-4)), bins=20)
ax[1].set_xlabel("log10(powierzchnia guza [mm²])")
ax[2].hist(m["tumor_frac_of_tissue"], bins=20); ax[2].set_xlabel("udział guza w tkance")
plt.tight_layout()
m.groupby("center")[["n_metastases", "tumor_mm2", "tissue_mm2", "tumor_frac_of_tissue"]].describe().T

## 6. Patche: ile jest jakich klas

Siatka patchy `PATCH`×`PATCH` px na poziomie 0, liczona na maskach z `MASK_LEVEL` (jeden patch = blok
`PATCH/downsample` px maski). Bierzemy tylko patche z ≥ 50% tkanki. Klasy: `guz` (≥ 50% patcha w guzie),
`normalny` (0% guza), `graniczny` (reszta; zwykle pomijany przy treningu klasyfikatora).

In [ ]:
def patch_table(tumor, tissue, ds, patch=PATCH):
    k = max(int(round(patch / ds)), 1)
    h, w = (tumor.shape[0] // k) * k, (tumor.shape[1] // k) * k
    t = tumor[:h, :w].reshape(h // k, k, w // k, k).mean(axis=(1, 3))
    s = tissue[:h, :w].reshape(h // k, k, w // k, k).mean(axis=(1, 3))
    r, c = np.nonzero(s >= 0.5)
    return pd.DataFrame(dict(row=r, col=c, tumor_frac=t[r, c], tissue_frac=s[r, c]))


tabs = []
for s in annotated:
    z = np.load(OUT / f"{s.slide_id}.npz")
    t = patch_table(z["tumor"], z["tissue"], float(z["downsample"]))
    tabs.append(t.assign(slide_id=s.slide_id, center=s.center, patient=s.patient))
patches = pd.concat(tabs, ignore_index=True)
patches["klasa"] = np.select(
    [patches["tumor_frac"] >= 0.5, patches["tumor_frac"] == 0], ["guz", "normalny"], "graniczny"
)
pd.crosstab(patches["center"], patches["klasa"], margins=True)

### Różnice barwienia między ośrodkami

Główne wyzwanie CAMELYON17 to przesunięcie dziedziny (domain shift): każdy ośrodek ma inny protokół barwienia
i skaner. Średni kolor tkanki na slajd (po jednym punkcie na slajd) powinien to pokazać.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for c, g in manifest.groupby("center"):
    ax[0].scatter(g["mean_r"], g["mean_b"], label=f"ośrodek {c}", s=40)
    ax[1].scatter(g["mean_g"], g["mean_b"], label=f"ośrodek {c}", s=40)
ax[0].set_xlabel("średni R tkanki"); ax[0].set_ylabel("średni B tkanki")
ax[1].set_xlabel("średni G tkanki"); ax[1].set_ylabel("średni B tkanki")
ax[0].legend()
plt.tight_layout()
manifest.groupby("center")[["mean_r", "mean_g", "mean_b"]].agg(["mean", "std"]).round(1)

## 7. Podział train / val / test

Slajdy tego samego pacjenta są silnie skorelowane, więc podział robimy **po pacjencie**, a nie po slajdzie,
żeby nie było przecieku. Przy pełnych danych losujemy osobno w każdym ośrodku (około 20% pacjentów na val i 20% na test, min. 1),
żeby val i test zawierały wszystkie rozkłady barwienia. Przy małym zbiorze (mniej niż 15 pacjentów, np. 5 albo 10 slajdów)
dzielimy globalnie po pacjencie (po ok. 20% na val i test, min. 1). Seed 0, wynik zamrożony w `configs/camelyon17_split.json`.
Jeśli plik już istnieje, jest wczytywany bez zmian. **Zmieniasz liczbę slajdów? Usuń ten plik** (oraz `tiles_manifest.csv`
i `tiles_params.json` w `data/processed/camelyon17`), żeby podział powstał od nowa.

In [ ]:
def make_split(manifest, seed=0, val_frac=0.2, test_frac=0.2):
    rng = np.random.default_rng(seed)
    val_p, test_p = set(), set()
    patients = sorted(manifest["patient"].unique())
    if len(patients) < 15:  # mały zbiór: podział globalny po pacjencie
        rng.shuffle(patients)
        if len(patients) >= 3:
            n_val = max(1, round(val_frac * len(patients)))
            n_test = max(1, round(test_frac * len(patients)))
            val_p = set(patients[:n_val])
            test_p = set(patients[n_val : n_val + n_test])
        elif len(patients) == 2:
            val_p = set(patients[:1])
    else:  # pełne dane: osobno w każdym ośrodku
        for c, g in manifest.groupby("center"):
            pats = sorted(g["patient"].unique())
            rng.shuffle(pats)
            if len(pats) >= 3:
                n_val = max(1, round(val_frac * len(pats)))
                n_test = max(1, round(test_frac * len(pats)))
                val_p |= set(pats[:n_val])
                test_p |= set(pats[n_val : n_val + n_test])
            elif len(pats) == 2:
                val_p |= set(pats[:1])
    is_val = manifest["patient"].isin(val_p)
    is_test = manifest["patient"].isin(test_p)
    return dict(seed=seed, group="patient",
                train=sorted(manifest.loc[~is_val & ~is_test, "slide_id"]),
                val=sorted(manifest.loc[is_val, "slide_id"]),
                test=sorted(manifest.loc[is_test, "slide_id"]))


if SPLIT_PATH.exists():
    split = json.loads(SPLIT_PATH.read_text())
else:
    split = make_split(manifest)
    SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    SPLIT_PATH.write_text(json.dumps(split, indent=2))

role = {i: "train" for i in split["train"]} | {i: "val" for i in split["val"]} | {i: "test" for i in split.get("test", [])}
mm = manifest.assign(role=manifest["slide_id"].map(role))
for a_, b_ in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not (set(mm.loc[mm.role == a_, "patient"]) & set(mm.loc[mm.role == b_, "patient"])), (a_, b_)
print({k: len(split.get(k, [])) for k in ("train", "val", "test")}, "slajdów")
print(mm.groupby("role")["tumor_px"].agg(["count", "sum", "mean"]))
pd.crosstab(mm["center"], mm["role"])

## Podsumowanie

Poniższa komórka drukuje najważniejsze liczby; wpisz je do notatek lub uzupełnij listę pod spodem.

In [ ]:
print(f"slajdy: {len(info)} ({info['patient'].nunique()} pacjentów, {info['center'].nunique()} ośrodków), "
      f"z adnotacją: {len(manifest)}")
print(f"poligony: {int(manifest['n_polygons'].sum())} "
      f"(guz: {int(manifest['n_metastases'].sum())}, normal: {int(manifest['n_normal'].sum())}); "
      f"odrzucone <3 wierzchołków: {int(manifest['n_too_few_vertices'].sum())}, "
      f"puste po rasteryzacji: {int(manifest['n_empty'].sum())}")
print("patche (≥50% tkanki):", patches["klasa"].value_counts().to_dict())
print(f"podział: {len(split['train'])} slajdów train, {len(split['val'])} val (po pacjencie, seed {split['seed']})")

- Zbiór treningowy CAMELYON17: 500 slajdów, 100 pacjentów, 5 ośrodków; adnotacje obszarów guza ma tylko część slajdów.
- Maski guza i tkanki leżą w `data/processed/camelyon17` (poziom `MASK_LEVEL`), podział w `configs/camelyon17_split.json`.
- Główne ryzyka: przesunięcie barwienia między ośrodkami, ogromna dysproporcja klas (guz to mały ułamek tkanki),
  przeciek między slajdami tego samego pacjenta (stąd podział po pacjencie).
- Następny krok: patche i oracle / inferencja na patchach.